# 02 SARIMA / SARIMAX Forecast

固定分割A/Bで、SARIMA と SARIMAX conditional forecast を最小実装で評価する。推定は log scale の `y` で行い、予測後に `exp()` で `number_parcels` スケールへ戻して評価する。

SARIMAX の外生変数は `baseline_m4` 仕様を使う。`covid_main` は 2020-03-01 から 2023-05-01 までとし、M5 は使わない。

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from statsmodels.tools.sm_exceptions import ConvergenceWarning


PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Transport_amount_project" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_connected_parcel_data
from src.forecasting.evaluation import evaluate_forecasts
from src.forecasting.sarimax import (
    fit_sarima,
    fit_sarimax,
    forecast_sarima,
    forecast_sarimax,
    prepare_exog_from_spec,
    run_sarima_grid_search,
    run_sarimax_grid_search,
    select_nonconstant_exog,
)
from src.forecasting.splits import make_fixed_split_a, make_fixed_split_b


DATA_PATH = PROJECT_ROOT / "data" / "processed" / "parcel_volume_connected.csv"
FORECAST_DIR = PROJECT_ROOT / "output" / "forecasts"
PREDICTIONS_DIR = FORECAST_DIR / "predictions"
METRICS_DIR = FORECAST_DIR / "metrics"
FIGURES_DIR = FORECAST_DIR / "figures"

for path in [PREDICTIONS_DIR, METRICS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

ORDER = (1, 1, 1)
SEASONAL_ORDER = (1, 1, 1, 12)
GRID_ORDERS = [(0, 1, 1), (1, 1, 0), (1, 1, 1), (2, 1, 1)]
GRID_SEASONAL_ORDERS = [(0, 1, 1, 12), (1, 1, 0, 12), (1, 1, 1, 12)]
SARIMAX_SPEC = "baseline_m4"

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Forecast output:", FORECAST_DIR)

## 1. データ読み込みと固定分割

接続済みデータを読み込み、fixed A/B を作成する。fixed B が作成できない場合はskipする設計にしているが、現在のデータは 2026-02 まであるため fixed B も実行できる。

In [ ]:
df = load_connected_parcel_data(str(DATA_PATH))

splits = {}
split_rows = []
for split_name, maker in [("fixed_a", make_fixed_split_a), ("fixed_b", make_fixed_split_b)]:
    try:
        split = maker(df)
        splits[split_name] = split
        split_rows.append(
            {
                "split": split_name,
                "status": "success",
                "train_start": split["train"].index.min().date(),
                "train_end": split["train"].index.max().date(),
                "train_rows": len(split["train"]),
                "test_start": split["test"].index.min().date(),
                "test_end": split["test"].index.max().date(),
                "test_rows": len(split["test"]),
                "message": "",
            }
        )
    except ValueError as exc:
        split_rows.append(
            {
                "split": split_name,
                "status": "skipped",
                "train_start": None,
                "train_end": None,
                "train_rows": 0,
                "test_start": None,
                "test_end": None,
                "test_rows": 0,
                "message": str(exc),
            }
        )

split_summary = pd.DataFrame(split_rows)
display(split_summary)

## 2. SARIMA / SARIMAX conditional の推定と予測

SARIMA は外生変数なし。SARIMAX は `baseline_m4` の外生変数を作成し、学習期間で分散がない列を自動除外する。fixed A では COVID 系と `post_stat_change` が学習期間で全て0になるため、SARIMAXでは主に学習期間内で変動する外生変数だけが残る。

In [ ]:
prediction_frames = []
fit_rows = []
used_exog_by_split = {}

for split_name, split in splits.items():
    train = split["train"]
    test = split["test"]
    print(f"Fitting {split_name} SARIMA/SARIMAX")

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        sarima_result = fit_sarima(train, order=ORDER, seasonal_order=SEASONAL_ORDER)
    sarima_retvals = getattr(sarima_result, "mle_retvals", {}) or {}
    sarima_warning = any(issubclass(w.category, ConvergenceWarning) for w in caught)
    sarima_forecast = forecast_sarima(sarima_result, test, split=split_name, spec_name="none")
    prediction_frames.append(sarima_forecast)
    fit_rows.append(
        {
            "model": "sarima",
            "split": split_name,
            "forecast_type": "unconditional",
            "spec_name": "none",
            "used_exog": "",
            "converged": sarima_retvals.get("converged"),
            "warnflag": sarima_retvals.get("warnflag"),
            "convergence_warning": sarima_warning,
            "log_likelihood": float(sarima_result.llf),
            "aic": float(sarima_result.aic),
            "bic": float(sarima_result.bic),
        }
    )

    exog_all = prepare_exog_from_spec(pd.concat([train, test]), SARIMAX_SPEC)
    exog_train = exog_all.loc[train.index]
    exog_test = exog_all.loc[test.index]
    exog_train_selected, exog_test_selected, used_exog = select_nonconstant_exog(exog_train, exog_test)
    used_exog_by_split[split_name] = used_exog

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        sarimax_result = fit_sarimax(
            train,
            exog_train_selected,
            order=ORDER,
            seasonal_order=SEASONAL_ORDER,
        )
    sarimax_retvals = getattr(sarimax_result, "mle_retvals", {}) or {}
    sarimax_warning = any(issubclass(w.category, ConvergenceWarning) for w in caught)
    sarimax_forecast = forecast_sarimax(
        sarimax_result,
        test,
        exog_test_selected,
        split=split_name,
        spec_name=SARIMAX_SPEC,
    )
    prediction_frames.append(sarimax_forecast)
    fit_rows.append(
        {
            "model": "sarimax",
            "split": split_name,
            "forecast_type": "conditional",
            "spec_name": SARIMAX_SPEC,
            "used_exog": ",".join(used_exog),
            "converged": sarimax_retvals.get("converged"),
            "warnflag": sarimax_retvals.get("warnflag"),
            "convergence_warning": sarimax_warning,
            "log_likelihood": float(sarimax_result.llf),
            "aic": float(sarimax_result.aic),
            "bic": float(sarimax_result.bic),
        }
    )

predictions_df = pd.concat(prediction_frames, ignore_index=True)
fit_summary = pd.DataFrame(fit_rows)
display(fit_summary)
display(pd.DataFrame([{"split": k, "used_exog": ",".join(v)} for k, v in used_exog_by_split.items()]))

## 3. 評価指標の計算

評価は `number_parcels` の原系列スケールで行う。既存の naive metrics があれば読み込み、seasonal naive と SARIMA/SARIMAX を同じ表で比較する。

In [ ]:
metrics_frames = []
for split_name, split in splits.items():
    split_predictions = predictions_df[predictions_df["split"] == split_name]
    split_metrics = evaluate_forecasts(
        split_predictions,
        y_train=split["train"]["number_parcels"],
    )
    metrics_frames.append(split_metrics)

sarimax_metrics = pd.concat(metrics_frames, ignore_index=True)

naive_metrics_path = METRICS_DIR / "naive_metrics.csv"
if naive_metrics_path.exists():
    naive_metrics = pd.read_csv(naive_metrics_path)
    comparison_metrics = pd.concat([naive_metrics, sarimax_metrics], ignore_index=True)
else:
    comparison_metrics = sarimax_metrics.copy()

display(sarimax_metrics)
display(comparison_metrics.sort_values(["split", "rmse"]))

## 4. 予測結果と評価指標の保存

分割ごとに SARIMA/SARIMAX の予測結果をまとめて保存し、評価指標は `sarimax_metrics.csv` に保存する。

In [ ]:
if "fixed_a" in splits:
    predictions_df[predictions_df["split"] == "fixed_a"].to_csv(PREDICTIONS_DIR / "fixed_a_sarimax.csv", index=False)

if "fixed_b" in splits:
    predictions_df[predictions_df["split"] == "fixed_b"].to_csv(PREDICTIONS_DIR / "fixed_b_sarimax.csv", index=False)

sarimax_metrics.to_csv(METRICS_DIR / "sarimax_metrics.csv", index=False)
fit_summary.to_csv(METRICS_DIR / "sarimax_fit_summary.csv", index=False)

print("Saved SARIMA/SARIMAX prediction and metric files.")

## 5. 予測図の保存

テスト期間の実績値と SARIMA/SARIMAX 予測を比較する。学習期間の最後の24か月も薄く表示する。

In [ ]:
def plot_split_forecasts(split_name: str, save_path: Path) -> None:
    split = splits[split_name]
    plot_df = predictions_df[predictions_df["split"] == split_name]

    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    train_tail = split["train"].tail(24)
    ax.plot(train_tail.index, train_tail["number_parcels"], color="0.55", linewidth=1.2, label="train actual tail")
    ax.plot(split["test"].index, split["test"]["number_parcels"], color="black", linewidth=1.6, label="test actual")

    for model_name, group in plot_df.groupby("model"):
        label = model_name
        if model_name == "sarimax":
            label = "sarimax conditional"
        ax.plot(group["date"], group["y_pred"], marker="o", linewidth=1.2, label=label)

    ax.axvline(split["train"].index.max(), color="0.2", linestyle=":", linewidth=1.0)
    ax.set_title(f"{split_name}: SARIMA/SARIMAX forecast comparison")
    ax.set_xlabel("Date")
    ax.set_ylabel("number_parcels")
    ax.grid(True, color="0.85", linewidth=0.8)
    ax.legend()
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


if "fixed_a" in splits:
    plot_split_forecasts("fixed_a", FIGURES_DIR / "fixed_a_sarimax_forecast.png")

if "fixed_b" in splits:
    plot_split_forecasts("fixed_b", FIGURES_DIR / "fixed_b_sarimax_forecast.png")

print("Saved SARIMA/SARIMAX forecast figures.")

## 6. Small grid search

The fixed `(1,1,1)(1,1,1,12)` SARIMAX forecasts were accurate, but convergence warnings appeared. This section tries a small manual grid for SARIMA and conditional SARIMAX on fixed A/B only. Rolling forecast and automatic order selection are intentionally outside this notebook.


In [ ]:
grid_rows = []

for split_name, split in splits.items():
    train = split["train"]
    test = split["test"]
    print(f"Grid search for {split_name}")

    sarima_grid = run_sarima_grid_search(
        train=train,
        test=test,
        split=split_name,
        orders=GRID_ORDERS,
        seasonal_orders=GRID_SEASONAL_ORDERS,
        spec_name="none",
    )
    grid_rows.append(sarima_grid)

    exog_all = prepare_exog_from_spec(pd.concat([train, test]), SARIMAX_SPEC)
    exog_train = exog_all.loc[train.index]
    exog_test = exog_all.loc[test.index]
    exog_train_selected, exog_test_selected, used_exog = select_nonconstant_exog(exog_train, exog_test)

    sarimax_grid = run_sarimax_grid_search(
        train=train,
        test=test,
        exog_train=exog_train_selected,
        exog_test=exog_test_selected,
        split=split_name,
        orders=GRID_ORDERS,
        seasonal_orders=GRID_SEASONAL_ORDERS,
        spec_name=SARIMAX_SPEC,
    )
    grid_rows.append(sarimax_grid)

grid_results = pd.concat(grid_rows, ignore_index=True)
display(grid_results.sort_values(["split", "model", "status", "rmse"]).head(20))


## 7. Best model selection

For each split and model family, converged candidates are preferred first. Among converged candidates, the best model is the one with the lowest RMSE. If no candidate converges, the lowest-RMSE candidate is kept with a caution note.


In [ ]:
best_rows = []
for (split_name, model_name), group in grid_results.groupby(["split", "model"]):
    valid = group[group["status"] == "success"].copy()
    converged = valid[valid["converged"] == True].copy()
    if not converged.empty:
        best = converged.sort_values("rmse").iloc[0].copy()
        best["selection_note"] = "best converged RMSE"
    elif not valid.empty:
        best = valid.sort_values("rmse").iloc[0].copy()
        best["selection_note"] = "no converged candidate; best RMSE with caution"
    else:
        best = group.iloc[0].copy()
        best["selection_note"] = "all candidates failed"
    best_rows.append(best)

best_models = pd.DataFrame(best_rows).reset_index(drop=True)
display(best_models[["split", "model", "order", "seasonal_order", "used_exog_columns", "converged", "warnflag", "rmse", "mae", "mape", "mase", "selection_note"]])


## 8. Save grid outputs

The full grid and best-model tables are saved under `output/forecasts/metrics/`. The RMSE comparison figure is saved under `output/forecasts/figures/`.


In [ ]:
grid_results.to_csv(METRICS_DIR / "sarimax_grid_search.csv", index=False)
best_models.to_csv(METRICS_DIR / "sarimax_grid_best_models.csv", index=False)

plot_grid = grid_results[grid_results["status"] == "success"].copy()
plot_grid["order_label"] = plot_grid["order"] + " / " + plot_grid["seasonal_order"]

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 8), sharex=True)
for ax, split_name in zip(axes, ["fixed_a", "fixed_b"]):
    split_grid = plot_grid[plot_grid["split"] == split_name].copy()
    labels = list(dict.fromkeys(split_grid["order_label"]))
    x = range(len(labels))
    for model_name, model_grid in split_grid.groupby("model"):
        values = [
            model_grid.loc[model_grid["order_label"] == label, "rmse"].min()
            for label in labels
        ]
        ax.plot(x, values, marker="o", linewidth=1.2, label=model_name)
    ax.set_title(f"{split_name}: grid RMSE")
    ax.set_ylabel("RMSE")
    ax.grid(True, color="0.85", linewidth=0.8)
    ax.legend()

axes[-1].set_xticks(list(x))
axes[-1].set_xticklabels(labels, rotation=45, ha="right")
axes[-1].set_xlabel("order / seasonal_order")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "sarimax_grid_rmse_comparison.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved grid search outputs.")


## 9. Reading the grid results

In this run, the grid provides converged SARIMAX candidates for both fixed A and fixed B, so it directly addresses the convergence warning seen in the fixed-order SARIMAX fit.

The interpretation should focus on the following points:

- Whether the convergence issue in the fixed-order SARIMAX fit improved.
- Whether SARIMA or conditional SARIMAX gives lower forecast error.
- Whether the grid candidates beat seasonal naive on RMSE.
- Whether fixed A and fixed B point to the same order or different orders.
- Which converged candidates are accurate enough to use as the next baseline.
